In [ ]:
import tensorflow as tf
import numpy as np
import pandas as pd
from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    AutoModelForSequenceClassification,
    AdamW,
    get_linear_schedule_with_warmup,
)
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score, confusion_matrix
import torch



In [ ]:
# Load Tokenizer
tokenizer = BertTokenizer.from_pretrained("ProsusAI/finbert")
tokenizer
# Tokenize the text columns
def tokenize_texts(text1):
    return tokenizer(text1.tolist(), padding='max_length', truncation=True, max_length=128)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

In [ ]:
sentiment_dict = {'neutral': 0, 'positive': 1, 'negative': 2}

In [ ]:
# Load the model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = BertForSequenceClassification.from_pretrained("ProsusAI/finbert", num_labels=len(sentiment_dict))
PATH = f"/content/8_finetunedCV_BERT_epoch_4.model"

model.load_state_dict(torch.load(PATH, map_location=torch.device('cpu')))
model.eval()

# model = BertForSequenceClassification.from_pretrained("ProsusAI/finbert", num_labels=len(sentiment_dict))
# PATH = f"/content/8_finetunedCV_BERT_epoch_4.model"

# model.load_state_dict(torch.load(PATH))
# model = model.to(device) # Set model to gpu


pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

<ipython-input-4-192291a688aa>:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(PATH, map_location=torch.device('cpu')))


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [ ]:
# Prediction method
def predict(texts):
    inputs = tokenizer(texts, padding='max_length', truncation=True, max_length=128, return_tensors="pt")
    with torch.no_grad():
        outputs = model(**inputs)
        predictions = torch.argmax(outputs.logits, dim=-1)
    return predictions

In [ ]:
# Load data for sentiment analysis
test_data = pd.read_csv(
    "Combine_Label_ActualValue_normalised.csv", encoding="UTF-8"
)

In [ ]:
test_data

,Unnamed: 0,Title,pubdate,Label,ActualValue,ActualDate,normalized_headline
0,0,"This Rally Has Legs, and Broad Reach, Too --- ...",2017-06-06,Positive,0,2017-06-06,"rally legs, broad reach, --- gains s&p 500 sho..."
1,1,"This Rally Has Legs, and Broad Reach, Too --- ...",2017-06-07,Positive,1,2017-06-07,"rally legs, broad reach, --- gains s&p 500 sho..."
2,2,U.S. Stock Futures Slip as U.K. Polls Suggest ...,2017-06-08,Negative,0,2017-06-08,u.s. stock futures slip u.k. polls suggest con...
3,3,Stocks Fall; Tech Shares Take Brunt; Tech stoc...,2017-06-12,Positive,1,2017-06-12,stocks fall; tech shares take brunt; tech stoc...
4,4,S&P 500 Slips After Fed Raises Interest Rates;...,2017-06-14,Negative,0,2017-06-14,s&p 500 slips fed raises interest rates; inves...
...,...,...,...,...,...,...,...
883,883,"Nvidia Pulls S&P 500, Nasdaq Lower; Stocks ros...",2024-06-24,Negative,0,2024-06-24,"nvidia pulls s&p 500, nasdaq lower; stocks ros..."
884,884,Dow Closes Higher; Nvidia Shares Extend Declin...,2024-06-24,Negative,0,2024-06-24,dow closes higher; nvidia shares extend declin...
885,885,"Nvidia Pulls S&P 500, Nasdaq Lower --- Stocks ...",2024-06-25,Negative,1,2024-06-25,"nvidia pulls s&p 500, nasdaq lower --- stocks ..."
886,886,"S&P 500, Nasdaq Extend Run of Gains; Walgreens...",2024-06-27,Positive,1,2024-06-27,"s&p 500, nasdaq extend run gains; walgreens si..."


In [ ]:
# Make a column for sentiment
test_data["Sentiment_Title"] = ""
print(len(test_data))
# Sentiment Analysis of data
for i in range(0, len(test_data)):
    text = test_data.loc[i, "Title"]
    predicted_classes = predict(text)

    if(predicted_classes[0] == 0):
      result_class = "Neutral"
    elif(predicted_classes[0] == 1):
      result_class = "Positive"
    else:
      result_class = "Negative"

    print(f"Predicted classes: {result_class}")
    test_data.loc[i, "Sentiment_Title"] = result_class

888
Predicted classes: Positive
Predicted classes: Positive
Predicted classes: Negative
Predicted classes: Positive
Predicted classes: Negative
Predicted classes: Negative
Predicted classes: Positive
Predicted classes: Positive
Predicted classes: Negative
Predicted classes: Positive
Predicted classes: Positive
Predicted classes: Negative
Predicted classes: Positive
Predicted classes: Negative
Predicted classes: Negative
Predicted classes: Positive
Predicted classes: Positive
Predicted classes: Positive
Predicted classes: Neutral
Predicted classes: Negative
Predicted classes: Negative
Predicted classes: Positive
Predicted classes: Positive
Predicted classes: Negative
Predicted classes: Negative
Predicted classes: Positive
Predicted classes: Positive
Predicted classes: Neutral
Predicted classes: Positive
Predicted classes: Negative
Predicted classes: Negative
Predicted classes: Positive
Predicted classes: Positive
Predicted classes: Positive
Predicted classes: Neutral
Predicted classes: 

In [ ]:
test_data

,Unnamed: 0,Title,pubdate,Label,ActualValue,ActualDate,normalized_headline,Sentiment_Title
0,0,"This Rally Has Legs, and Broad Reach, Too --- ...",2017-06-06,Positive,0,2017-06-06,"rally legs, broad reach, --- gains s&p 500 sho...",Positive
1,1,"This Rally Has Legs, and Broad Reach, Too --- ...",2017-06-07,Positive,1,2017-06-07,"rally legs, broad reach, --- gains s&p 500 sho...",Positive
2,2,U.S. Stock Futures Slip as U.K. Polls Suggest ...,2017-06-08,Negative,0,2017-06-08,u.s. stock futures slip u.k. polls suggest con...,Negative
3,3,Stocks Fall; Tech Shares Take Brunt; Tech stoc...,2017-06-12,Positive,1,2017-06-12,stocks fall; tech shares take brunt; tech stoc...,Positive
4,4,S&P 500 Slips After Fed Raises Interest Rates;...,2017-06-14,Negative,0,2017-06-14,s&p 500 slips fed raises interest rates; inves...,Negative
...,...,...,...,...,...,...,...,...
883,883,"Nvidia Pulls S&P 500, Nasdaq Lower; Stocks ros...",2024-06-24,Negative,0,2024-06-24,"nvidia pulls s&p 500, nasdaq lower; stocks ros...",Positive
884,884,Dow Closes Higher; Nvidia Shares Extend Declin...,2024-06-24,Negative,0,2024-06-24,dow closes higher; nvidia shares extend declin...,Negative
885,885,"Nvidia Pulls S&P 500, Nasdaq Lower --- Stocks ...",2024-06-25,Negative,1,2024-06-25,"nvidia pulls s&p 500, nasdaq lower --- stocks ...",Positive
886,886,"S&P 500, Nasdaq Extend Run of Gains; Walgreens...",2024-06-27,Positive,1,2024-06-27,"s&p 500, nasdaq extend run gains; walgreens si...",Positive


In [ ]:
# Make a column for sentiment
test_data["Sentiment_PreprocessedTitle"] = ""
print(len(test_data))
# Sentiment Analysis of preprocessed data
for i in range(0, len(test_data)):
    text = test_data.loc[i, "normalized_headline"]
    predicted_classes = predict(text)

    if(predicted_classes[0] == 0):
      result_class = "Neutral"
    elif(predicted_classes[0] == 1):
      result_class = "Positive"
    else:
      result_class = "Negative"

    print(f"Predicted classes: {result_class}")
    test_data.loc[i, "Sentiment_PreprocessedTitle"] = result_class

888
Predicted classes: Neutral
Predicted classes: Neutral
Predicted classes: Negative
Predicted classes: Negative
Predicted classes: Neutral
Predicted classes: Neutral
Predicted classes: Positive
Predicted classes: Positive
Predicted classes: Neutral
Predicted classes: Positive
Predicted classes: Positive
Predicted classes: Negative
Predicted classes: Positive
Predicted classes: Negative
Predicted classes: Negative
Predicted classes: Positive
Predicted classes: Positive
Predicted classes: Positive
Predicted classes: Neutral
Predicted classes: Negative
Predicted classes: Negative
Predicted classes: Positive
Predicted classes: Positive
Predicted classes: Negative
Predicted classes: Neutral
Predicted classes: Positive
Predicted classes: Positive
Predicted classes: Positive
Predicted classes: Positive
Predicted classes: Positive
Predicted classes: Negative
Predicted classes: Positive
Predicted classes: Positive
Predicted classes: Positive
Predicted classes: Neutral
Predicted classes: Posit

In [ ]:
test_data

,Unnamed: 0,Title,pubdate,Label,ActualValue,ActualDate,normalized_headline,Sentiment_Title,Sentiment_PreprocessedTitle
0,0,"This Rally Has Legs, and Broad Reach, Too --- ...",2017-06-06,Positive,0,2017-06-06,"rally legs, broad reach, --- gains s&p 500 sho...",Positive,Neutral
1,1,"This Rally Has Legs, and Broad Reach, Too --- ...",2017-06-07,Positive,1,2017-06-07,"rally legs, broad reach, --- gains s&p 500 sho...",Positive,Neutral
2,2,U.S. Stock Futures Slip as U.K. Polls Suggest ...,2017-06-08,Negative,0,2017-06-08,u.s. stock futures slip u.k. polls suggest con...,Negative,Negative
3,3,Stocks Fall; Tech Shares Take Brunt; Tech stoc...,2017-06-12,Positive,1,2017-06-12,stocks fall; tech shares take brunt; tech stoc...,Positive,Negative
4,4,S&P 500 Slips After Fed Raises Interest Rates;...,2017-06-14,Negative,0,2017-06-14,s&p 500 slips fed raises interest rates; inves...,Negative,Neutral
...,...,...,...,...,...,...,...,...,...
883,883,"Nvidia Pulls S&P 500, Nasdaq Lower; Stocks ros...",2024-06-24,Negative,0,2024-06-24,"nvidia pulls s&p 500, nasdaq lower; stocks ros...",Positive,Positive
884,884,Dow Closes Higher; Nvidia Shares Extend Declin...,2024-06-24,Negative,0,2024-06-24,dow closes higher; nvidia shares extend declin...,Negative,Negative
885,885,"Nvidia Pulls S&P 500, Nasdaq Lower --- Stocks ...",2024-06-25,Negative,1,2024-06-25,"nvidia pulls s&p 500, nasdaq lower --- stocks ...",Positive,Positive
886,886,"S&P 500, Nasdaq Extend Run of Gains; Walgreens...",2024-06-27,Positive,1,2024-06-27,"s&p 500, nasdaq extend run gains; walgreens si...",Positive,Negative


In [ ]:
test_data.to_csv("Result_CaseModel.csv")

In [ ]:
result_all_data = pd.read_csv("Result_CaseModel_rvNeutral.csv")

In [ ]:
result_all_data

,Unnamed: 0.1,Unnamed: 0,Title,pubdate,Label,ActualValue,ActualDate,normalized_headline,Sentiment_Title,Sentiment_PreprocessedTitle,Number_Sent_Title,Number_Sent_Preprocessed
0,0,0,"This Rally Has Legs, and Broad Reach, Too --- ...",2017-06-06,Positive,0,2017-06-06,"rally legs, broad reach, --- gains s&p 500 sho...",Positive,Negative,1,0
1,1,1,"This Rally Has Legs, and Broad Reach, Too --- ...",2017-06-07,Positive,1,2017-06-07,"rally legs, broad reach, --- gains s&p 500 sho...",Positive,Negative,1,0
2,2,2,U.S. Stock Futures Slip as U.K. Polls Suggest ...,2017-06-08,Negative,0,2017-06-08,u.s. stock futures slip u.k. polls suggest con...,Negative,Negative,0,0
3,3,3,Stocks Fall; Tech Shares Take Brunt; Tech stoc...,2017-06-12,Positive,1,2017-06-12,stocks fall; tech shares take brunt; tech stoc...,Positive,Positive,1,1
4,4,4,S&P 500 Slips After Fed Raises Interest Rates;...,2017-06-14,Negative,0,2017-06-14,s&p 500 slips fed raises interest rates; inves...,Negative,Positive,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...
883,883,883,"Nvidia Pulls S&P 500, Nasdaq Lower; Stocks ros...",2024-06-24,Negative,0,2024-06-24,"nvidia pulls s&p 500, nasdaq lower; stocks ros...",Negative,Negative,0,0
884,884,884,Dow Closes Higher; Nvidia Shares Extend Declin...,2024-06-24,Negative,0,2024-06-24,dow closes higher; nvidia shares extend declin...,Negative,Negative,0,0
885,885,885,"Nvidia Pulls S&P 500, Nasdaq Lower --- Stocks ...",2024-06-25,Negative,1,2024-06-25,"nvidia pulls s&p 500, nasdaq lower --- stocks ...",Negative,Negative,0,0
886,886,886,"S&P 500, Nasdaq Extend Run of Gains; Walgreens...",2024-06-27,Positive,1,2024-06-27,"s&p 500, nasdaq extend run gains; walgreens si...",Positive,Positive,1,1
